In [2]:
import pandas as pd
import polars as pl
from pathlib import Path

In [3]:
raw_data_dir = Path.cwd().parent / 'data_gather' / 'raw_data'
filtered_data_dir = Path.cwd().parent / 'data_gather' / 'filtered_data'


In [ ]:
trades = pl.scan_csv(raw_data_dir / 'coinbase_trades.csv').collect()
options = pl.scan_csv(raw_data_dir / 'deribit_option_vols.csv').collect()
kalshi = pl.scan_csv(raw_data_dir / 'kalshi_markets.csv').collect()
polymarket = pl.scan_csv(raw_data_dir / 'polymarket_markets.csv').collect()

In [20]:
filterer = filter_data(['BTC', 'ETH'])
df = filterer.clean_trades(trades)

In [ ]:
dfz = df.with_columns(
    pl.col("curr_time")
    .str.to_datetime(format="%Y-%m-%d %H:%M:%S%.f%#z", strict=False)
    .alias("curr_time"))

dfz.head()

id,curr_time,product_id,trade_time,price,size,side,curr_time_readable
u32,"datetime[μs, UTC]",str,str,f64,f64,str,str
0,2026-03-21 10:47:31.458355 UTC,"""BTC-USD""","""2026-03-21 10:47:31.433808+00:…",70565.43,0.000084,"""sell""","""2026-03-21 06:47:31"""
1,2026-03-21 10:47:31.573966 UTC,"""BTC-USD""","""2026-03-21 10:47:31.554989+00:…",70565.43,4.4000e-7,"""sell""","""2026-03-21 06:47:31"""
2,2026-03-21 10:47:31.705201 UTC,"""ETH-USD""","""2026-03-21 10:47:31.697621+00:…",2153.74,0.000556,"""sell""","""2026-03-21 06:47:31"""
3,2026-03-21 10:47:32.041475 UTC,"""ETH-USD""","""2026-03-21 10:47:32.029096+00:…",2153.73,0.000596,"""buy""","""2026-03-21 06:47:32"""
4,2026-03-21 10:47:32.041505 UTC,"""ETH-USD""","""2026-03-21 10:47:32.029096+00:…",2153.73,0.1502702,"""buy""","""2026-03-21 06:47:32"""


In [34]:
class filter_data(): 
    def __init__(self
                 , trades: pl.DataFrame
                 , options: pl.DataFrame
                 , kalshi: pl.DataFrame
                 , polymarket: pl.DataFrame):
        
        self.trades = trades
        self.options = options
        self.kalshi = kalshi
        self.polymarket = polymarket
    
    def clean_trades(self, coins: list) -> pl.DataFrame:
        coin_trades = self.trades.filter(pl.col('product_id').is_in([f'{coin}-USD' for coin in coins]))
        coin_trades = coin_trades.unique(subset=['trade_id'])
        coin_trades = coin_trades.sort('trade_time', descending=False)
        coin_trades = coin_trades.drop('id').with_row_index('id')
        coin_trades = coin_trades.select('id', 'curr_time', 'product_id', 'trade_time', 'price', 'size', 'side')
        return coin_trades
    
    def clean_options(self, coins: list) -> pl.DataFrame: 
        coin_options = self.options.filter(pl.col('currency').is_in([f'{coin}' for coin in coins]))
        coin_options = coin_options.sort('curr_time', descending=False)
        coin_options = coin_options.drop('id').with_row_index('id')
        coin_options = coin_options.select('id', 'curr_time', 'instrument_name', 'expiry_datetime', 'strike', 
                                           'option_type', 'underlying_price', 'delta', 'mark_iv', 'mark_price', 
                                           'open_interest', 'volume')
        return coin_options
    
    def clean_kalshi(self, coins: list) -> pl.DataFrame: 
        coin_kalshi = self.kalshi.filter(pl.col('coin').is_in([f'{coin}' for coin in coins]))
        coin_kalshi = coin_kalshi.sort('curr_time', descending=False)
        coin_kalshi = coin_kalshi.drop('id').with_row_index('id')
        coin_kalshi = coin_kalshi.select('id', 'curr_time', 'coin', 'open_time', 'close_time', 
                                         'last_price_dollars' , 'no_ask_dollars', 'no_bid_dollars', 
                                         'yes_ask_dollars', 'yes_bid_dollars', 'floor_strike', 'volume_24h_fp')
        return coin_kalshi
    
    def clean_polymarket(self, coins: list) -> pl.DataFrame:
        coin_polymarket = self.polymarket.filter(pl.col('coin').is_in([f'{coin}' for coin in coins]))
        coin_polymarket = coin_polymarket.sort('curr_time', descending=False)
        coin_polymarket = coin_polymarket.drop('id').with_row_index('id')
        coin_polymarket = coin_polymarket.select('id', 'curr_time', 'coin', 'interval_start_unix', 'end_date', 'strike_price',
                                                 'liquidity', 'volume', 'yes_implied_price', 'no_implied_price', 'yes_buy_price',
                                                 'yes_sell_price', 'no_buy_price', 'no_sell_price')
        return coin_polymarket
    
        

In [37]:
FILTERER = filter_data(trades=trades, options=options, kalshi=kalshi, polymarket=polymarket)


In [40]:
btc_trades = FILTERER.clean_trades(['BTC']).write_csv(filtered_data_dir / 'btc_trades.csv')
btc_eth_options = FILTERER.clean_options(['BTC', 'ETH']).write_csv(filtered_data_dir / 'btc_eth_options.csv')
all_coins_kalshi = FILTERER.clean_kalshi(['BTC', 'ETH', 'XRP', 'SOL']).write_csv(filtered_data_dir / 'all_kalshi.csv')
all_coins_polymarket = FILTERER.clean_polymarket(['BTC', 'ETH', 'XRP', 'SOL']).write_csv(filtered_data_dir / 'all_polymarket.csv')

In [ ]:
clean_options = options.filter(pl.col('option_type') == 'call').drop('option_type').with_row_index('id')

In [30]:
btc_trades = btc_trades.with_columns(
    ((pl.col("price") * pl.col("size")) * pl.when(pl.col("side") == "buy").then(1).otherwise(-1)).alias("weighted")
)
btc_trades['weighted'].sum()


-42187385.281208575

In [ ]:
options.head()


id,curr_time,currency,instrument_name,expiry_datetime,strike,option_type,underlying_price,delta,mark_iv,bid_price,ask_price,mark_price,open_interest,volume
i64,str,str,str,str,f64,str,f64,f64,f64,str,str,f64,f64,f64
1120902,"""2026-03-21 16:47:26.155308+00:…","""ETH""","""ETH-22MAR26-2125-P""","""2026-03-22 00:00:00+00:00""",2125.0,"""P""",2149.8527,-0.19584,32.83,null,null,0.0015,7364.0,6292.0
1120903,"""2026-03-21 16:47:31.156183+00:…","""BTC""","""BTC-22MAR26-71000-C""","""2026-03-22 00:00:00+00:00""",71000.0,"""C""",70399.3669,0.21469,25.64,null,null,0.0013,84.9,82.9
1120904,"""2026-03-21 16:47:31.156183+00:…","""BTC""","""BTC-22MAR26-70500-C""","""2026-03-22 00:00:00+00:00""",70500.0,"""C""",70399.3669,0.4472,24.86,null,null,0.0035,222.4,134.8
1120905,"""2026-03-21 16:47:31.156183+00:…","""BTC""","""BTC-22MAR26-70000-P""","""2026-03-22 00:00:00+00:00""",70000.0,"""P""",70399.3259,-0.2967,25.84,null,null,0.002,561.6,189.5
1120906,"""2026-03-21 16:47:31.156183+00:…","""ETH""","""ETH-22MAR26-2175-C""","""2026-03-22 00:00:00+00:00""",2175.0,"""C""",2149.9038,0.19826,32.6,null,null,0.0015,1176.0,890.0


In [ ]:
kalshi.head()


id,curr_time,coin,open_time,close_time,last_price_dollars,liquidity_dollars,no_ask_dollars,no_bid_dollars,yes_ask_dollars,yes_bid_dollars,open_interest,floor_strike,cap_strike,strike_type,volume,volume_24h_fp
i64,str,str,str,str,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,f64
4200710,"""2026-03-23 06:21:15.231426+00:…","""XRP""","""2026-03-23 06:15:00+00:00""","""2026-03-23 06:30:00+00:00""",0.52,0.0,0.5,0.48,0.52,0.5,null,1.3815,null,"""greater_or_equal""",null,0.0
4200711,"""2026-03-23 06:21:15.231426+00:…","""SOL""","""2026-03-23 06:15:00+00:00""","""2026-03-23 06:30:00+00:00""",0.58,0.0,0.51,0.47,0.53,0.49,null,86.5482,null,"""greater_or_equal""",null,0.0
4200712,"""2026-03-23 06:21:16.232547+00:…","""BTC""","""2026-03-23 06:15:00+00:00""","""2026-03-23 06:30:00+00:00""",0.62,0.0,0.38,0.37,0.63,0.62,null,68569.81,null,"""greater_or_equal""",null,0.0
4200713,"""2026-03-23 06:21:16.232547+00:…","""ETH""","""2026-03-23 06:15:00+00:00""","""2026-03-23 06:30:00+00:00""",0.55,0.0,0.5,0.46,0.54,0.5,null,2057.469999,null,"""greater_or_equal""",null,0.0
4200714,"""2026-03-23 06:21:16.232547+00:…","""XRP""","""2026-03-23 06:15:00+00:00""","""2026-03-23 06:30:00+00:00""",0.52,0.0,0.5,0.48,0.52,0.5,null,1.3815,null,"""greater_or_equal""",null,0.0


In [ ]:
polymarket.head()

id,curr_time,coin,time_horizon_minutes,interval_start_unix,market_slug,condition_id,strike_price,end_date,liquidity,volume,open_interest,yes_implied_price,no_implied_price,yes_buy_price,yes_sell_price,no_buy_price,no_sell_price
i64,str,str,i64,i64,str,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64
0,"""2026-03-28 18:00:42.725060+00:…","""ETH""",15,1774720800,"""eth-updown-15m-1774720800""","""0xd9b38938db1e4c03d96213a63a73…",2019.45,"""2026-03-28 18:15:00+00:00""",38862.5508,155.27818,null,0.54,0.46,0.47,0.48,0.52,0.53
1,"""2026-03-28 18:00:42.725060+00:…","""XRP""",15,1774720800,"""xrp-updown-15m-1774720800""","""0xf49dd149e33579a84c9ea9dfbaad…",1.3462,"""2026-03-28 18:15:00+00:00""",34937.5142,63.995962,null,0.575,0.425,0.45,0.46,0.54,0.55
2,"""2026-03-28 18:00:42.725060+00:…","""SOL""",15,1774720800,"""sol-updown-15m-1774720800""","""0x8be6b4663170642c83a1d69fe5a9…",83.17,"""2026-03-28 18:15:00+00:00""",36528.5169,42.22,null,0.535,0.465,0.49,0.5,0.5,0.51
3,"""2026-03-28 18:00:43.735854+00:…","""BTC""",15,1774720800,"""btc-updown-15m-1774720800""","""0xe2debe513a849862fee606de1680…",66853.8,"""2026-03-28 18:15:00+00:00""",48924.7393,817.438353,null,0.475,0.525,0.54,0.55,0.45,0.46
4,"""2026-03-28 18:00:43.735854+00:…","""ETH""",15,1774720800,"""eth-updown-15m-1774720800""","""0xd9b38938db1e4c03d96213a63a73…",2019.45,"""2026-03-28 18:15:00+00:00""",38862.5508,155.27818,null,0.54,0.46,0.47,0.48,0.52,0.53
